# GELAR RASA 2026 — Processed Train, GPU Training, Final Submission

Notebook ini mempertahankan **assignment kategori, source group, crop, dan pengecualian processed aktif** dari alur utama. Seluruh piksel dibuat ulang dari dataset GelarRasa yang terpasang di Kaggle. Snapshot metadata algoritmis disertakan secara eksplisit di dalam notebook; tidak ada file label atau cache lokal yang perlu dipasang.

**Asal label:** snapshot hasil proses train-only sebelumnya (termasuk 37 seed yang ditinjau tim). Notebook ini **tidak mengulang fitting seluruh metode clustering/pelabelan historis**. Ia memverifikasi hash sumber dan menerapkan ulang preprocessing. Hash piksel dibandingkan persis terlebih dahulu; perbedaan kecil lintas runtime diperiksa dengan ringkasan warna blok 16x16 sebelum training. Metadata test berisi identitas dan hash, tanpa target kategori.

Target port aktif: **1.879 foto utuh + 7 crop = 1.886 sampel train**, 27 turunan train ditahan, dan 480 gambar test. Angka 1.869 merupakan versi processed terdahulu. Training di notebook ini memakai seluruh 1.886 sampel eligible; tidak ada filter confidence atau pseudo-labeler baru yang mengganti assignment.

Keluaran: delapan folder processed train, processed test, manifest dan bukti kesamaan, EDA/preview, fold/OOF, riwayat serta checkpoint GPU, probabilitas per backbone, dan `DSC2026_Submission.csv`. Metrik OOF mengukur agreement terhadap assignment algoritmis yang dibekukan, bukan ground truth independen.

Versi hybrid v3 menambahkan jalur DINOv3/SVM/group-kNN serta Extra Trees dan XGBoost. Kecocokan dengan CSV pembanding tidak digunakan oleh notebook untuk fitting atau seleksi. Hasil target test tidak dijamin; kesamaan processed tetap diperiksa sebelum training.


In [ ]:
import os,sys,json,time,random,hashlib,platform,zipfile,shutil,gc,base64,zlib
from pathlib import Path
from datetime import datetime,timezone
import numpy as np,pandas as pd
from PIL import Image,ImageOps,ImageFilter
import matplotlib.pyplot as plt
from IPython.display import display
import torch,torchvision
import importlib.metadata as package_metadata
SEED=2026
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
def seed_everything(value):
    random.seed(value); np.random.seed(value); torch.manual_seed(value)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(value)
seed_everything(SEED)
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True
torch.use_deterministic_algorithms(True,warn_only=True)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type!='cuda': raise RuntimeError('Aktifkan GPU pada Session options Kaggle sebelum training.')
INPUT_ROOT=Path('/kaggle/input'); WORK_ROOT=Path('/kaggle/working/gelarrasa_run')
for d in ['eda','processed','labels','validation','checkpoints','predictions','logs','figures']:
    (WORK_ROOT/d).mkdir(parents=True,exist_ok=True)
GPU_INFO=[{'name':torch.cuda.get_device_name(i),'memory_GB':round(torch.cuda.get_device_properties(i).total_memory/1024**3,2)} for i in range(torch.cuda.device_count())]
ENVIRONMENT={'python':sys.version,'platform':platform.platform(),'torch':torch.__version__,'torchvision':torchvision.__version__,'pillow':package_metadata.version('Pillow'),'numpy':np.__version__,'pandas':pd.__version__,'cuda':torch.version.cuda,'gpu':GPU_INFO,'seed':SEED}
print('Pipeline: active_processed_hybrid_v3')
print(json.dumps(ENVIRONMENT,indent=2))
(WORK_ROOT/'logs/environment.json').write_text(json.dumps(ENVIRONMENT,indent=2),encoding='utf-8')

## 1. Deteksi dan integritas input

Jalur Kaggle pada log sebelumnya didukung langsung. Fallback mencari pasangan train/test dan sample CSV yang dapat berada pada tingkat folder berbeda. Keunikan ID diperiksa per split; nomor train dan test boleh sama. Nilai kategori contoh pada sample CSV diabaikan.

In [ ]:
# Cari train/test dan sample CSV secara terpisah: CSV bisa berada di parent folder.
IMAGE_EXTS={'.jpg','.jpeg','.png','.webp','.bmp'}
SEARCH_ROOTS=[Path('/kaggle/input'),Path('/kaggle/working'),Path('/kaggle/temp'),Path('/content'),Path('/mnt/data')]
SEARCH_ROOTS=[p for p in SEARCH_ROOTS if p.exists()]
def walk_inventory(roots):
    dirs=[]; files=[]
    for root in roots:
        for base,subdirs,names in os.walk(root,followlinks=True):
            subdirs.sort(); here=Path(base); dirs.append(here)
            files.extend(here/name for name in names)
    return dirs,files

def is_split_dir(p,split):
    name=p.name.lower().replace(' ','_').replace('-','_')
    return name==split or name==split+'ing' or name.startswith(split+'_') or name.startswith(split+'images')

def find_layout(roots):
    dirs,files=walk_inventory(roots)
    trains=[p for p in dirs if is_split_dir(p,'train')]
    tests=[p for p in dirs if is_split_dir(p,'test')]
    samples=[p for p in files if p.suffix.lower()=='.csv' and 'sample' in p.name.lower() and 'submission' in p.name.lower()]
    options=[]
    for tr in trains:
        for te in tests:
            for sm in samples:
                try: root=Path(os.path.commonpath([str(tr),str(te),str(sm.parent)]))
                except ValueError: continue
                sibling=0 if tr.parent==te.parent else 1
                # Prefer sibling image folders and the deepest shared dataset directory.
                options.append((sibling,-len(root.parts),str(root),root,tr,te,sm))
    if not options: return None,dirs,files
    options.sort(key=lambda x:(x[0],x[1],x[2]))
    _,_,_,root,tr,te,sm=options[0]
    return (root,tr,te,sm),dirs,files

# Jalur langsung terlihat pada log runtime Kaggle saat ini.
_KAGGLE_ROOT=Path('/kaggle/input/datasets/arielzzzzzz/gelarrasa')
_KAGGLE_IMAGE_ROOT=_KAGGLE_ROOT/'Dataset DSC GelarRasa 2026'
_KAGGLE_TRAIN=_KAGGLE_IMAGE_ROOT/'train'; _KAGGLE_TEST=_KAGGLE_IMAGE_ROOT/'test'
_KAGGLE_SAMPLE=_KAGGLE_ROOT/'SampleSubmission_DSC.csv'
if _KAGGLE_TRAIN.is_dir() and _KAGGLE_TEST.is_dir() and _KAGGLE_SAMPLE.is_file():
    layout=(_KAGGLE_ROOT,_KAGGLE_TRAIN,_KAGGLE_TEST,_KAGGLE_SAMPLE)
    seen_dirs,seen_files=[],[]
    print('Dataset ditemukan pada path Kaggle yang diketahui:',_KAGGLE_ROOT)
else:
    layout,seen_dirs,seen_files=find_layout(SEARCH_ROOTS)
if layout is None:
    zip_files=[p for p in seen_files if p.suffix.lower()=='.zip']
    if zip_files:
        unpack=Path('/kaggle/temp/gelarrasa_dataset'); unpack.mkdir(parents=True,exist_ok=True); base=unpack.resolve()
        for z in zip_files:
            with zipfile.ZipFile(z) as f:
                for m in f.infolist():
                    dest=(unpack/m.filename).resolve()
                    if base not in dest.parents and dest!=base: raise RuntimeError(f'Unsafe ZIP path: {m.filename}')
                f.extractall(unpack)
        layout,seen_dirs,seen_files=find_layout([unpack])
if layout is None:
    details=[]
    for d in seen_dirs:
        try: depth=min(len(d.relative_to(root).parts) for root in SEARCH_ROOTS if d==root or root in d.parents)
        except ValueError: continue
        if depth<=5:
            try: entries=list(d.iterdir())
            except OSError: entries=[]
            details.append({'path':str(d),'dirs':[x.name for x in entries if x.is_dir()][:20],'files':[x.name for x in entries if x.is_file()][:20]})
    raise FileNotFoundError('Dataset folders not found. Scanned '+str([str(x) for x in SEARCH_ROOTS])+'; visible structure: '+json.dumps(details,ensure_ascii=False))
DATA_ROOT,TRAIN_ROOT,TEST_ROOT,SAMPLE_PATH=layout

def image_files(folder):
    direct=[x for x in folder.iterdir() if x.is_file() and x.suffix.lower() in IMAGE_EXTS]
    files=direct or [x for x in folder.rglob('*') if x.is_file() and x.suffix.lower() in IMAGE_EXTS]
    return sorted(files,key=lambda x:(x.stem,x.as_posix()))
def norm_id(x): return str(x).strip().split('.')[0].zfill(4)
train_paths=image_files(TRAIN_ROOT); test_paths=image_files(TEST_ROOT)
train_ids=[norm_id(x.stem) for x in train_paths]; test_ids=[norm_id(x.stem) for x in test_paths]
sample=pd.read_csv(SAMPLE_PATH,dtype={'image_id':str})
assert list(sample.columns)==['image_id','waste_category'] and sample.image_id.is_unique, f'Sample schema invalid: {sample.columns.tolist()}'
sample['image_id_norm']=sample.image_id.map(norm_id)
assert len(sample)==len(test_paths), f'Test file count {len(test_paths)} != sample rows {len(sample)}; test folder: {TEST_ROOT}'
assert set(sample.image_id_norm)==set(test_ids), f'Test/sample IDs mismatch. Extra test: {sorted(set(test_ids)-set(sample.image_id_norm))[:10]}; extra sample: {sorted(set(sample.image_id_norm)-set(test_ids))[:10]}'
# ID boleh sama antar-split karena train memakai 4 digit dan test/sample 3 digit; cek unik per split saja.
assert len(set(train_ids))==len(train_ids), 'Duplicate IDs inside train split.'
assert len(set(test_ids))==len(test_ids), 'Duplicate IDs inside test split.'
CLASSES=['cardboard','electronic','foodwaste','glass','metal','paper','plastic','textile']
print('DATA_ROOT:',DATA_ROOT,'| TRAIN_ROOT:',TRAIN_ROOT,'| TEST_ROOT:',TEST_ROOT,'| SAMPLE_PATH:',SAMPLE_PATH)
print('train:',len(train_paths),'| test:',len(test_paths),'| classes:',len(CLASSES))
# Ignore waste_category values in the sample submission.

raw_train_paths=list(train_paths); raw_test_paths=list(test_paths)
raw_train_ids=list(train_ids); raw_test_ids=list(test_ids)

## 2. EDA sumber dan kualitas citra

Inventaris semua sumber sebelum transformasi: hash file/piksel, dimensi, rasio, ukuran byte dan indikator edge variance. Identitas gambar berasal dari pasangan split dan ID, bukan angka ID lintas split. Kesamaan piksel train-test merupakan audit deskriptif.

In [ ]:
def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''): h.update(b)
    return h.hexdigest()
def profile(p,split):
    r={'split':split,'image_id':norm_id(p.stem),'file_name':p.name,'file_sha256':sha256_file(p),'bytes':p.stat().st_size}
    with Image.open(p) as src:
        im=ImageOps.exif_transpose(src); rgb=im.convert('RGB'); gray=np.asarray(rgb.convert('L'),dtype=np.float32)
        r.update(width=rgb.width,height=rgb.height,mode=im.mode,aspect_ratio=rgb.width/rgb.height,
                 pixel_sha256=hashlib.sha256(rgb.tobytes()).hexdigest(),
                 blur_edge_variance=float(np.asarray(rgb.convert('L').filter(ImageFilter.FIND_EDGES),dtype=np.float32).var()))
    return r
manifest=pd.DataFrame([profile(p,'train') for p in train_paths]+[profile(p,'test') for p in test_paths])
manifest.to_csv(WORK_ROOT/'eda/image_manifest.csv',index=False)
summary=manifest.groupby('split').agg(n=('image_id','size'),median_width=('width','median'),median_height=('height','median'),median_bytes=('bytes','median'),median_aspect=('aspect_ratio','median'),median_edge_variance=('blur_edge_variance','median')).reset_index()
summary.to_csv(WORK_ROOT/'eda/image_summary.csv',index=False)
a=set(manifest.query("split=='train'").pixel_sha256); b=set(manifest.query("split=='test'").pixel_sha256)
(WORK_ROOT/'eda/exact_duplicate_summary.json').write_text(json.dumps({'train_test_exact_pixel_overlap':len(a&b),'use':'diagnostic only'},indent=2),encoding='utf-8')
print(summary.to_string(index=False))
fig,axs=plt.subplots(1,3,figsize=(15,4))
for split,g in manifest.groupby('split'):
    axs[0].hist(g.width,bins=30,alpha=.5,label=split); axs[1].hist(g.height,bins=30,alpha=.5,label=split); axs[2].hist(g.aspect_ratio,bins=30,alpha=.5,label=split)
for ax,title in zip(axs,['Width','Height','Aspect ratio']): ax.set_title(title); ax.legend()
plt.tight_layout(); plt.savefig(WORK_ROOT/'figures/image_dimensions.png',dpi=150); plt.show()
def contact_sheet(paths,title,out,n=24):
    selected=paths[:n]; cols=6; rows=(len(selected)+cols-1)//cols
    fig,axs=plt.subplots(rows,cols,figsize=(15,rows*2.7)); axs=np.array(axs).reshape(-1)
    for ax,p in zip(axs,selected):
        with Image.open(p) as im: ax.imshow(ImageOps.exif_transpose(im).convert('RGB'))
        ax.set_title(norm_id(p.stem)); ax.axis('off')
    for ax in axs[len(selected):]: ax.axis('off')
    fig.suptitle(title); plt.tight_layout(); plt.savefig(out,dpi=130); plt.show()
contact_sheet(train_paths,'Train samples',WORK_ROOT/'figures/train_contact_sheet.png')
contact_sheet(test_paths,'Test preview',WORK_ROOT/'figures/test_contact_sheet.png')

## 3. Snapshot assignment dan kebijakan processed aktif

Blok terkompresi berikut hanya berisi metadata sumber, assignment train yang sudah dibekukan, source group, bounding box crop, status eligible, dan hash hasil processed. Tidak memuat piksel, target test, submission pembanding, atau fitur model historis. Hash payload dan hash semua sumber diverifikasi sebelum materialisasi. Snapshot berasal dari manifest operasional alur utama, bukan hasil classifier EfficientNet 37-seed yang dipakai pada versi Kaggle sebelumnya.

In [ ]:
import base64,zlib
FROZEN_STATE_B64=(
'...'
)
STATE_SHA256='f8928662997b1909abf8dd212f90b9d4d91261698053971096d2d779f27d3cef'
state_bytes=zlib.decompress(base64.b64decode(FROZEN_STATE_B64))
assert hashlib.sha256(state_bytes).hexdigest()==STATE_SHA256,'Snapshot metadata corrupted.'
PROCESS_STATE=json.loads(state_bytes)
CLASSES=PROCESS_STATE['classes']
print('Snapshot:',PROCESS_STATE['version'],'| SHA256:',STATE_SHA256)
print('Expected population:',PROCESS_STATE['expected_counts'])
print('Frozen assignment origins:',pd.Series([r['label_source'] for r in PROCESS_STATE['rows'] if r['split']=='train' and r['training_eligible']]).value_counts().to_dict())
(WORK_ROOT/'labels/processing_state.json').write_text(json.dumps(PROCESS_STATE,indent=2),encoding='utf-8')

## 4. Buat ulang processed dan periksa kesamaannya

Terapkan RGB/EXIF, crop native sesuai koordinat, lalu `ImageOps.contain` ke sisi maksimum 202 dan padding RGB `(127,127,127)` pada kanvas 224×224. Foto utuh/crop eligible disimpan pada delapan folder kelas; turunan yang ditahan terpisah di review. Semua test diproses tanpa label. Hash piksel setiap hasil dibandingkan; jika berbeda, ringkasan warna blok 16x16 harus tetap dekat. Perbedaan visual besar menghentikan proses sebelum training.

In [ ]:
def file_sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1<<20),b''): h.update(block)
    return h.hexdigest()
def rgb_source(path):
    with Image.open(path) as source:
        im=ImageOps.exif_transpose(source)
        if 'A' in im.getbands() or 'transparency' in im.info:
            rgba=im.convert('RGBA')
            im=Image.alpha_composite(Image.new('RGBA',rgba.size,(255,255,255,255)),rgba)
        return im.convert('RGB').copy()
def processed_canvas(im,policy):
    content=ImageOps.contain(im,(policy['max_content_side'],policy['max_content_side']),method=Image.Resampling.LANCZOS)
    size=policy['output_size']; out=Image.new('RGB',(size,size),tuple(policy['padding_rgb']))
    out.paste(content,((size-content.width)//2,(size-content.height)//2))
    return out
def thumbnail16(canvas):
    pixels=np.asarray(canvas,dtype=np.uint8)
    assert pixels.shape==(224,224,3),'Unexpected processed canvas size.'
    return np.rint(pixels.reshape(16,14,16,14,3).mean(axis=(1,3))).astype(np.uint8)
def validate_processed_pixels(canvas,spec):
    actual_sha=hashlib.sha256(canvas.tobytes()).hexdigest()
    exact=actual_sha==spec['expected_pixel_sha256']
    expected=np.frombuffer(zlib.decompress(base64.b64decode(spec['expected_thumbnail16_b64'])),dtype=np.uint8).reshape(16,16,3)
    actual=thumbnail16(canvas)
    error=np.abs(actual.astype(np.int16)-expected.astype(np.int16))
    mean_error=float(error.mean()); p99_error=float(np.percentile(error,99))
    assert exact or (mean_error<=2.0 and p99_error<=15.0), (
        f'Processed image differs materially: {spec["split"]}:{spec["sample_id"]}; '
        f'thumbnail MAE={mean_error:.3f}, p99={p99_error:.1f}')
    return actual_sha,exact,mean_error,p99_error
def materialize_processed(state,raw_train,raw_test,output_root):
    paths={'train':list(raw_train),'test':list(raw_test)}
    lookup={(split,p.name):p for split,pp in paths.items() for p in pp}
    expected={(r['split'],r['file_name']) for r in state['sources']}
    assert set(lookup)==expected,'Raw source names/count differ from frozen processed source dataset.'
    source_checks=[]
    for s in state['sources']:
        p=lookup[(s['split'],s['file_name'])]; sha=file_sha256(p)
        assert sha==s['sha256'],f'Raw file SHA256 mismatch: {s["split"]}/{s["file_name"]}'
        source_checks.append({**s,'actual_sha256':sha,'matches':True})
    for name in state['classes']: (output_root/'train'/name).mkdir(parents=True,exist_ok=True)
    out_rows=[]
    for spec in state['rows']:
        p=lookup[(spec['split'],spec['source_file'])]; im=rgb_source(p)
        x0,y0,x1,y1=spec['bbox']
        assert 0<=x0<x1<=im.width and 0<=y0<y1<=im.height,f'Invalid crop: {spec["sample_id"]}'
        region=im.crop((x0,y0,x1,y1)); canvas=processed_canvas(region,state['policy'])
        pixel_sha,pixel_exact,thumbnail_mae,thumbnail_p99=validate_processed_pixels(canvas,spec)
        if spec['split']=='test': dest=output_root/'test'/f'{spec["sample_id"]}.png'
        else:
            branch='train' if spec['training_eligible'] else 'review/excluded_train'
            dest=output_root/branch/spec['assigned_class']/f'{spec["sample_id"]}.png'
        dest.parent.mkdir(parents=True,exist_ok=True); canvas.save(dest,format='PNG')
        native=''
        if spec['is_crop']:
            native=output_root/'crops'/f'{spec["sample_id"]}.png'; native.parent.mkdir(parents=True,exist_ok=True); region.save(native,format='PNG')
        out_rows.append({**spec,'source_path':str(p),'output_path':str(dest),'native_crop_path':str(native),'actual_pixel_sha256':pixel_sha,'pixel_matches_frozen':pixel_exact,'thumbnail_mae':thumbnail_mae,'thumbnail_p99':thumbnail_p99})
    # Reserve source is retained outside all classifier folders.
    for parent in state['policy']['reserve_review']:
        matches=[p for p in raw_train if norm_id(p.stem)==parent]
        assert len(matches)==1
        dest=output_root/'review/reserved'/f'{parent}.png'; dest.parent.mkdir(parents=True,exist_ok=True); rgb_source(matches[0]).save(dest)
    return pd.DataFrame(out_rows),pd.DataFrame(source_checks)

PROCESSED_ROOT=WORK_ROOT/'processed'
processed,source_checks=materialize_processed(PROCESS_STATE,raw_train_paths,raw_test_paths,PROCESSED_ROOT)
processed.to_csv(PROCESSED_ROOT/'manifest.csv',index=False)
source_checks.to_csv(WORK_ROOT/'eda/source_integrity.csv',index=False)
train_frame=processed[processed.split.eq('train') & processed.training_eligible].reset_index(drop=True)
test_frame=processed[processed.split.eq('test')].reset_index(drop=True)
expected=PROCESS_STATE['expected_counts']
assert len(train_frame)==expected['eligible_train'] and len(test_frame)==expected['processed_test']
assert int(train_frame.is_crop.sum())==expected['eligible_crops']
assert train_frame.assigned_class.value_counts().to_dict()==PROCESS_STATE['class_counts']
assert not test_frame.assigned_class.astype(str).str.len().any()
reviewed=train_frame[train_frame.human_reviewed]
assert reviewed.assigned_class.value_counts().le(5).all() and len(reviewed)<=40
assert reviewed.parent_id.is_unique
assert train_frame.sample_id.is_unique and test_frame.sample_id.is_unique
train_paths=[Path(p) for p in train_frame.output_path]; test_paths=[Path(p) for p in test_frame.output_path]
train_ids=train_frame.sample_id.tolist(); test_ids=test_frame.parent_id.map(norm_id).tolist()
pseudo=np.array([CLASSES.index(c) for c in train_frame.assigned_class],dtype=np.int64)
eligible_idx=np.arange(len(train_frame)); groups=train_frame.parent_group.astype(str).to_numpy()
# Reuse the frozen source groups; all crops of a source inherit its group.
assert train_frame.groupby('parent_id').parent_group.nunique().le(1).all()
assert set(test_ids)==set(sample.image_id_norm) and len(test_ids)==len(sample)
train_frame.to_csv(WORK_ROOT/'labels/train_targets.csv',index=False)
proof={'state_sha256':STATE_SHA256,'source_files_verified':len(source_checks),'processed_pixels_verified':len(processed),'exact_pixel_matches':int(processed.pixel_matches_frozen.sum()),'thumbnail_tolerance_matches':int((~processed.pixel_matches_frozen).sum()),'max_thumbnail_mae':float(processed.thumbnail_mae.max()),'max_thumbnail_p99':float(processed.thumbnail_p99.max()),'eligible_train':len(train_frame),'eligible_full_frame':int((~train_frame.is_crop).sum()),'eligible_crops':int(train_frame.is_crop.sum()),'train_source_groups':int(train_frame.parent_group.nunique()),'class_counts':train_frame.assigned_class.value_counts().to_dict(),'labels_recomputed':False,'labels_origin':PROCESS_STATE['origin'],'all_pixels_match_active_processed':bool(processed.pixel_matches_frozen.all()),'all_images_passed_visual_integrity':True}
(WORK_ROOT/'processed/equivalence_report.json').write_text(json.dumps(proof,indent=2),encoding='utf-8')
print(json.dumps(proof,indent=2))

## 5. Distribusi akhir, preview crop, dan audit kesamaan

Hasil berikut menampilkan himpunan yang benar-benar dibaca model. Tujuh crop dipreview bersama gambar induknya; kelompok source tetap melekat pada crop. Histogram berasal dari manifest processed, bukan kategori sample submission.

In [ ]:
counts=train_frame.assigned_class.value_counts().reindex(CLASSES,fill_value=0)
counts.rename('train_samples').to_csv(WORK_ROOT/'labels/train_distribution.csv')
display(counts.to_frame('train_samples'))
fig,ax=plt.subplots(figsize=(10,4)); counts.plot.bar(ax=ax,rot=25,title='Distribusi train processed aktif'); ax.set_ylabel('sampel'); plt.tight_layout(); plt.savefig(WORK_ROOT/'figures/processed_train_distribution.png',dpi=150); plt.show(); plt.close(fig)
fig,axs=plt.subplots(2,4,figsize=(14,7))
for ax,category in zip(axs.flat,CLASSES):
    row=train_frame[train_frame.assigned_class.eq(category)].iloc[0]
    ax.imshow(rgb_source(row.output_path)); ax.set_title(f'{category} | {row.sample_id}'); ax.axis('off')
plt.tight_layout(); plt.savefig(WORK_ROOT/'figures/processed_class_preview.png',dpi=150); plt.show(); plt.close(fig)
crops=train_frame[train_frame.is_crop]
fig,axs=plt.subplots(len(crops),2,figsize=(8,3*len(crops)),squeeze=False)
for axes,row in zip(axs,crops.itertuples()):
    axes[0].imshow(rgb_source(row.source_path)); axes[0].set_title(f'Train source {row.parent_id}')
    axes[1].imshow(rgb_source(row.output_path)); axes[1].set_title(f'{row.sample_id}: {row.assigned_class}')
    for ax in axes: ax.axis('off')
plt.tight_layout(); plt.savefig(WORK_ROOT/'figures/crop_preprocessing_preview.png',dpi=130); plt.show(); plt.close(fig)
print('Low-confidence exclusion from previous Kaggle version is disabled; frozen eligible rows retained:',len(train_frame))

## 6. Kemiripan processed train-test (diagnostik)

Backbone beku EfficientNetV2-S hanya menghitung representasi deskriptif untuk pencarian kemiripan. Fitur test tidak dipakai fit normalisasi, PCA, target, sampler, atau model. Hasil nearest-train tidak mengubah satu pun assignment; CNN berikut memakai target processed yang sudah diverifikasi.

In [ ]:
from torch.utils.data import Dataset,DataLoader,WeightedRandomSampler
from torchvision import models,transforms
from torchvision.transforms import InterpolationMode
MEAN=(.485,.456,.406); STD=(.229,.224,.225); MODEL_IMAGE_SIZE=224
class ImageTensor(Dataset):
    def __init__(self,paths,augment=False):
        self.paths=list(paths); ops=[transforms.Resize((MODEL_IMAGE_SIZE,MODEL_IMAGE_SIZE),interpolation=InterpolationMode.BICUBIC)]
        if augment:
            ops += [transforms.RandomHorizontalFlip(.5),transforms.RandomAffine(8,translate=(.03,.03),scale=(.93,1.07),interpolation=InterpolationMode.BILINEAR,fill=(127,127,127)),transforms.RandomApply([transforms.ColorJitter(.12,.12,.12,.025)],p=.4)]
        self.tf=transforms.Compose(ops+[transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
    def __len__(self): return len(self.paths)
    def __getitem__(self,i):
        with Image.open(self.paths[i]) as im: return self.tf(im.convert('RGB'))
try: embedding_net=models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1).to(DEVICE).eval()
except Exception as error: raise RuntimeError('Bobot pretrained gagal diunduh. Aktifkan Internet Kaggle; bobot acak tidak digunakan.') from error
def extract_embeddings(paths):
    loader=DataLoader(ImageTensor(paths),batch_size=24,shuffle=False,num_workers=2,pin_memory=True)
    chunks=[]
    with torch.inference_mode(),torch.autocast(device_type=DEVICE.type,enabled=DEVICE.type=='cuda',dtype=torch.float16):
        for xb in loader:
            xb=xb.to(DEVICE,non_blocking=True); z=embedding_net.avgpool(embedding_net.features(xb)).flatten(1)
            chunks.append(z.float().cpu().numpy())
    result=np.concatenate(chunks).astype('float32'); result/=np.maximum(np.linalg.norm(result,axis=1,keepdims=True),1e-12)
    return result
train_emb=extract_embeddings(train_paths); test_emb=extract_embeddings(test_paths)
np.savez_compressed(WORK_ROOT/'eda/processed_embeddings.npz',train=train_emb,test=test_emb,train_ids=np.array(train_ids),test_ids=np.array(test_ids))
similarity=test_emb@train_emb.T
nearest=np.argsort(-similarity,axis=1)[:,:3]
from scipy.fft import dctn
from skimage.metrics import structural_similarity
def pixel_descriptor(path):
    with Image.open(path) as im:
        gray=im.convert('L')
        small=np.asarray(gray.resize((32,32),Image.Resampling.LANCZOS),dtype=np.float32)
        low=dctn(small,type=2,norm='ortho')[:8,:8]; bits=(low>np.median(low)).reshape(-1)
        pix=np.asarray(gray.resize((64,64),Image.Resampling.LANCZOS),dtype=np.float32)/255.
    return bits,pix
train_desc=[pixel_descriptor(p) for p in train_paths]; test_desc=[pixel_descriptor(p) for p in test_paths]
train_phash=np.stack([p[0] for p in train_desc]); test_phash=np.stack([p[0] for p in test_desc])
train_pixels=np.stack([p[1] for p in train_desc]); test_pixels=np.stack([p[1] for p in test_desc])
neighbor_rows=[]
for q,nn in enumerate(nearest):
    for rank,j in enumerate(nn,1):
        neighbor_rows.append({'test_id':test_ids[q],'rank':rank,'train_sample_id':train_ids[j],'train_class':train_frame.iloc[j].assigned_class,'cosine':float(similarity[q,j]),'phash_hamming':int(np.count_nonzero(test_phash[q]!=train_phash[j])),'grayscale_mae_64':float(np.abs(test_pixels[q]-train_pixels[j]).mean()),'ssim_gray_64':float(structural_similarity(test_pixels[q],train_pixels[j],data_range=1.)),'train_group':groups[j],'train_native_width':int(train_frame.iloc[j].bbox[2]-train_frame.iloc[j].bbox[0]),'train_native_height':int(train_frame.iloc[j].bbox[3]-train_frame.iloc[j].bbox[1]),'test_native_width':int(test_frame.iloc[q].bbox[2]-test_frame.iloc[q].bbox[0]),'test_native_height':int(test_frame.iloc[q].bbox[3]-test_frame.iloc[q].bbox[1])})
pd.DataFrame(neighbor_rows).to_csv(WORK_ROOT/'eda/processed_train_test_neighbors.csv',index=False)
phash_nearest=[]
for q in range(len(test_paths)):
    distance=np.count_nonzero(train_phash!=test_phash[q],axis=1); j=int(distance.argmin())
    phash_nearest.append({'test_id':test_ids[q],'train_sample_id':train_ids[j],'phash_hamming':int(distance[j]),'embedding_cosine':float(similarity[q,j]),'candidate_near_duplicate':bool(distance[j]<=4)})
pd.DataFrame(phash_nearest).to_csv(WORK_ROOT/'eda/processed_phash_nearest.csv',index=False)
query_positions=np.argsort(-similarity.max(axis=1))[:8]
fig,axs=plt.subplots(len(query_positions),2,figsize=(8,3*len(query_positions)),squeeze=False)
for axes,q in zip(axs,query_positions):
    j=int(nearest[q,0]); axes[0].imshow(rgb_source(test_paths[q])); axes[0].set_title(f'Test {test_ids[q]} (no target)')
    axes[1].imshow(rgb_source(train_paths[j])); axes[1].set_title(f'Train {train_ids[j]}: {train_frame.iloc[j].assigned_class} | cos {similarity[q,j]:.3f}')
    for ax in axes: ax.axis('off')
plt.tight_layout(); plt.savefig(WORK_ROOT/'figures/processed_train_test_neighbors.png',dpi=130); plt.show(); plt.close(fig)
del embedding_net; gc.collect(); torch.cuda.empty_cache()

## 7. Source-group cross-validation dan GPU training

Semua training memakai assignment processed yang sudah dibekukan. Tiga fold dikelompokkan memakai source group yang sama seperti alur utama; crop dan induknya tidak dapat terpisah lintas fold. Setiap model menjalani 8 epoch/fold, dengan checkpoint terbaik berdasarkan macro-F1 proxy. Refit final memakai 3 epoch head + 18 epoch fine-tuning dan menyimpan bobot pada akhir jadwal tersebut. Final refit tidak memiliki validation set sehingga log final tidak mengklaim metrik validation.

Empat backbone memakai AMP dan kedua GPU jika tersedia, class-balanced sampler, MixUp pada piksel input, label smoothing, dan cosine learning rate. Tiap epoch menyimpan state optimizer/scheduler/scaler/RNG untuk resume pada working directory yang sama. Resume hanya berlaku bila fingerprint processed dan konfigurasi cocok.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score,f1_score,classification_report,confusion_matrix,log_loss
from torch.nn import functional as F
MODEL_NAMES=['resnet50','efficientnet_v2_s','convnext_small','swin_v2_t']
HEAD_EPOCHS=3; FINETUNE_EPOCHS=18; FOLD_EPOCHS=8; N_SPLITS=3; BATCH_SIZE=24; WORKERS=2
USE_MULTI_GPU=True; RUN_START=time.time()
TRAIN_CONFIG={'models':MODEL_NAMES,'head_epochs':HEAD_EPOCHS,'finetune_epochs':FINETUNE_EPOCHS,'fold_epochs':FOLD_EPOCHS,'folds':N_SPLITS,'batch_size':BATCH_SIZE,'model_size':MODEL_IMAGE_SIZE,'seed':SEED,'state_sha256':STATE_SHA256,'torch':str(torch.__version__),'torchvision':str(torchvision.__version__)}
CONFIG_HASH=hashlib.sha256(json.dumps(TRAIN_CONFIG,sort_keys=True).encode()).hexdigest()
(WORK_ROOT/'logs/train_config.json').write_text(json.dumps(TRAIN_CONFIG,indent=2),encoding='utf-8')
def core_model(model): return model.module if isinstance(model,torch.nn.DataParallel) else model
def get_head(model,name):
    m=core_model(model)
    if name=='resnet50': return m.fc
    if name=='efficientnet_v2_s': return m.classifier[1]
    if name=='convnext_small': return m.classifier[2]
    if name=='swin_v2_t': return m.head
    raise ValueError(f'Unknown model: {name}')
def build_model(name,pretrained=True):
    if name=='resnet50':
        m=models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None); m.fc=torch.nn.Linear(m.fc.in_features,len(CLASSES))
    elif name=='efficientnet_v2_s':
        m=models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1 if pretrained else None); m.classifier[1]=torch.nn.Linear(m.classifier[1].in_features,len(CLASSES))
    elif name=='convnext_small':
        m=models.convnext_small(weights=models.ConvNeXt_Small_Weights.IMAGENET1K_V1 if pretrained else None); m.classifier[2]=torch.nn.Linear(m.classifier[2].in_features,len(CLASSES))
    elif name=='swin_v2_t':
        m=models.swin_v2_t(weights=models.Swin_V2_T_Weights.IMAGENET1K_V1 if pretrained else None); m.head=torch.nn.Linear(m.head.in_features,len(CLASSES))
    else: raise ValueError(name)
    return m
class LabeledDataset(Dataset):
    def __init__(self,indices,augment=False):
        self.indices=np.asarray(indices); self.base=ImageTensor([train_paths[j] for j in self.indices],augment)
    def __len__(self): return len(self.indices)
    def __getitem__(self,k):
        j=int(self.indices[k]); return self.base[k],int(pseudo[j]),train_ids[j]
def worker_seed(worker_id):
    value=torch.initial_seed()%2**32; np.random.seed(value); random.seed(value)
def make_train_loader(indices,seed):
    counts=np.bincount(pseudo[indices],minlength=len(CLASSES)); weights=1/np.maximum(counts,1)
    generator=torch.Generator().manual_seed(seed)
    sampler=WeightedRandomSampler(torch.tensor(weights[pseudo[indices]],dtype=torch.double),num_samples=len(indices),replacement=True,generator=generator)
    return DataLoader(LabeledDataset(indices,True),batch_size=BATCH_SIZE,sampler=sampler,num_workers=WORKERS,pin_memory=DEVICE.type=='cuda',worker_init_fn=worker_seed,generator=generator)
def predict_loader(model,loader):
    model.eval(); chunks=[]; labels=[]; ids=[]
    with torch.inference_mode(),torch.autocast(device_type=DEVICE.type,enabled=DEVICE.type=='cuda',dtype=torch.float16):
        for batch in loader:
            xb=batch[0] if isinstance(batch,(list,tuple)) else batch
            chunks.append(model(xb.to(DEVICE,non_blocking=True)).float().softmax(1).cpu().numpy())
            if isinstance(batch,(list,tuple)):
                labels.extend(batch[1].numpy().tolist()); ids.extend(list(batch[2]))
    return np.concatenate(chunks),labels,ids
def train_epoch(model,loader,opt,scaler,head_only,name):
    model.train()
    if head_only: model.eval(); get_head(model,name).train()
    loss_sum=0.; n=0
    for xb,yb,_ in loader:
        xb=xb.to(DEVICE,non_blocking=True); yb=yb.to(DEVICE); opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type,enabled=DEVICE.type=='cuda',dtype=torch.float16):
            if np.random.random()<.3 and len(xb)>1:
                lam=float(np.random.beta(.15,.15)); perm=torch.randperm(len(xb),device=DEVICE)
                logits=model(lam*xb+(1-lam)*xb[perm])
                loss=lam*F.cross_entropy(logits,yb,label_smoothing=.025)+(1-lam)*F.cross_entropy(logits,yb[perm],label_smoothing=.025)
            else: loss=F.cross_entropy(model(xb),yb,label_smoothing=.025)
        scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),2.0); scaler.step(opt); scaler.update()
        loss_sum+=float(loss.detach().cpu())*len(yb); n+=len(yb)
    return loss_sum/max(n,1)
def load_checkpoint(path): return torch.load(path,map_location='cpu',weights_only=False)
def atomic_save(obj,path):
    tmp=path.with_suffix('.tmp'); torch.save(obj,tmp); os.replace(tmp,path)
def fit_one(name,tr_indices,va_indices,epochs,tag):
    seed=SEED+MODEL_NAMES.index(name)*100+sum(tag.encode()); seed_everything(seed)
    best_path=WORK_ROOT/f'checkpoints/{tag}_{name}_best.pt'
    final_path=WORK_ROOT/f'checkpoints/{tag}_{name}_final.pt'
    resume_path=WORK_ROOT/f'checkpoints/{tag}_{name}_resume.pt'
    done_path=WORK_ROOT/f'checkpoints/{tag}_{name}_done.json'
    chosen=best_path if va_indices is not None else final_path
    if done_path.exists() and chosen.exists():
        info=json.loads(done_path.read_text()); assert info['config_hash']==CONFIG_HASH,'Existing completed model has a different configuration.'
        ckpt=load_checkpoint(chosen); model=build_model(name,False); model.load_state_dict(ckpt['state']); model=model.to(DEVICE)
        if USE_MULTI_GPU and DEVICE.type=='cuda' and torch.cuda.device_count()>1: model=torch.nn.DataParallel(model)
        print('Reusing completed model:',tag,name); return model,pd.read_csv(WORK_ROOT/f'logs/{tag}_{name}_history.csv').to_dict('records'),chosen
    resume=load_checkpoint(resume_path) if resume_path.exists() else None
    if resume: assert resume['config_hash']==CONFIG_HASH,'Resume data/config mismatch.'
    model=build_model(name,pretrained=resume is None)
    if resume: model.load_state_dict(resume['state'])
    model=model.to(DEVICE)
    if USE_MULTI_GPU and DEVICE.type=='cuda' and torch.cuda.device_count()>1: model=torch.nn.DataParallel(model)
    scaler=torch.amp.GradScaler(DEVICE.type,enabled=DEVICE.type=='cuda')
    history=resume['history'] if resume else []; completed=int(resume['completed']) if resume else 0; best=float(resume['best']) if resume else -1.
    if resume:
        random.setstate(resume['rng_python']); np.random.set_state(resume['rng_numpy']); torch.set_rng_state(resume['rng_torch'])
        if DEVICE.type=='cuda': torch.cuda.set_rng_state_all(resume['rng_cuda'])
        scaler.load_state_dict(resume['scaler'])
    val_loader=DataLoader(LabeledDataset(va_indices),batch_size=BATCH_SIZE*2,shuffle=False,num_workers=WORKERS,pin_memory=DEVICE.type=='cuda') if va_indices is not None else None
    stages=[('head',min(HEAD_EPOCHS,epochs)),('finetune',max(0,epochs-HEAD_EPOCHS))]
    start=0
    for stage,count in stages:
        if count==0: continue
        stop=start+count
        if completed>=stop: start=stop; continue
        for param in model.parameters(): param.requires_grad=stage!='head'
        for param in get_head(model,name).parameters(): param.requires_grad=True
        head_params=list(get_head(model,name).parameters()); head_ids={id(p) for p in head_params}
        if stage=='head': param_groups=[{'params':head_params,'lr':1e-3}]
        else: param_groups=[{'params':[p for p in model.parameters() if id(p) not in head_ids],'lr':2e-5},{'params':head_params,'lr':1e-4}]
        opt=torch.optim.AdamW(param_groups,weight_decay=1e-4); scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=count,eta_min=1e-6)
        if resume and resume['stage']==stage and completed>start:
            opt.load_state_dict(resume['optimizer']); scheduler.load_state_dict(resume['scheduler'])
            # Optimizer tensors are moved explicitly for CPU-map-location resume.
            for values in opt.state.values():
                for key,value in values.items():
                    if torch.is_tensor(value): values[key]=value.to(DEVICE)
        for epoch in range(max(completed,start),stop):
            epoch_start=time.time(); loader=make_train_loader(tr_indices,seed+epoch)
            loss=train_epoch(model,loader,opt,scaler,stage=='head',name)
            row={'model':name,'tag':tag,'stage':stage,'epoch':epoch+1,'train_loss':loss,'lr_backbone_or_head':float(opt.param_groups[0]['lr']),'seconds':round(time.time()-epoch_start,2)}
            if val_loader is not None:
                pr,yy,_=predict_loader(model,val_loader); prediction=pr.argmax(1)
                row.update(val_proxy_agreement=float(accuracy_score(yy,prediction)),val_proxy_macro_f1=float(f1_score(yy,prediction,labels=list(range(len(CLASSES))),average='macro',zero_division=0)),val_proxy_logloss=float(log_loss(yy,pr,labels=list(range(len(CLASSES))))))
                if row['val_proxy_macro_f1']>best:
                    best=row['val_proxy_macro_f1']; atomic_save({'state':core_model(model).state_dict(),'name':name,'epoch':epoch+1,'config_hash':CONFIG_HASH,'classes':CLASSES},best_path)
            scheduler.step(); history.append(row); completed=epoch+1
            pd.DataFrame(history).to_csv(WORK_ROOT/f'logs/{tag}_{name}_history.csv',index=False)
            atomic_save({'state':core_model(model).state_dict(),'name':name,'stage':stage,'completed':completed,'best':best,'history':history,'optimizer':opt.state_dict(),'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),'rng_python':random.getstate(),'rng_numpy':np.random.get_state(),'rng_torch':torch.get_rng_state(),'rng_cuda':torch.cuda.get_rng_state_all() if DEVICE.type=='cuda' else [],'config_hash':CONFIG_HASH},resume_path)
            print(row,flush=True)
        start=stop
    if va_indices is None: atomic_save({'state':core_model(model).state_dict(),'name':name,'epoch':completed,'config_hash':CONFIG_HASH,'classes':CLASSES},final_path)
    model.load_state_dict({('module.'+k if isinstance(model,torch.nn.DataParallel) else k):v for k,v in load_checkpoint(chosen)['state'].items()})
    done_path.write_text(json.dumps({'config_hash':CONFIG_HASH,'epochs':completed,'checkpoint':str(chosen)}),encoding='utf-8')
    if resume_path.exists(): resume_path.unlink()
    return model,history,chosen

splitter=StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=SEED)
folds=list(splitter.split(np.zeros(len(train_frame)),pseudo,groups)); fold_ids=np.full(len(train_frame),-1)
for fold,(tr,va) in enumerate(folds):
    assert set(groups[tr]).isdisjoint(set(groups[va]))
    assert set(pseudo[tr])==set(range(len(CLASSES)))
    fold_ids[va]=fold
assert (fold_ids>=0).all()
fold_table=train_frame[['sample_id','parent_id','parent_group','assigned_class','is_crop']].copy(); fold_table['fold']=fold_ids; fold_table.to_csv(WORK_ROOT/'validation/folds.csv',index=False)

def write_submission(probabilities,path):
    assert probabilities.shape==(len(test_ids),len(CLASSES)) and np.isfinite(probabilities).all()
    assert np.allclose(probabilities.sum(1),1,atol=1e-4)
    pred=pd.DataFrame({'image_id_norm':test_ids,'waste_category':[CLASSES[j] for j in probabilities.argmax(1)]})
    out=sample[['image_id','image_id_norm']].merge(pred,on='image_id_norm',how='left',validate='one_to_one',sort=False)
    assert out.image_id.tolist()==sample.image_id.tolist()
    assert len(out)==480 and out.waste_category.notna().all() and set(out.waste_category)<=set(CLASSES)
    out=out[['image_id','waste_category']]; out.to_csv(path,index=False)
    return out

## 8. DINOv3 dan fitur warna/bentuk

Jalur yang sudah digunakan pada eksperimen sebelumnya: DINOv3 Base image-only, checkpoint/revisi/hash dibekukan, descriptor CLS + patch valid + foreground mean + foreground std. Anggota train, label, group dan ROI tetap persis processed aktif. View native ROI dipakai sebelum resize model agar resep fitur lama terjaga. Test hanya ditransformasi dengan encoder beku. HSV/Lab/spatial color, HOG dan LBP menambah informasi warna/bentuk bagi tree models.

In [ ]:
import importlib.metadata,subprocess,joblib
DEPENDENCY_PINS={'timm':'1.0.27','xgboost':'3.1.2'}
for package,required in DEPENDENCY_PINS.items():
    try: current=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: current=None
    if current!=required:
        subprocess.check_call([sys.executable,'-m','pip','install','--quiet',f'{package}=={required}'])
import timm,xgboost,cv2
from huggingface_hub import hf_hub_download
from sklearn.preprocessing import normalize
from skimage.feature import hog,local_binary_pattern
D3_REPOSITORY='timm/vit_base_patch16_dinov3.lvd1689m'
D3_REVISION='c6a5fb7d12bbd3cf3b0079253141c3332aaed7da'
D3_WEIGHT_SHA256='1f9ed8a2378d65e24bb710ba522ac9fa7be4e036d7aefb4384ce022833926332'
D3_RECIPE='dinov3base_256_letterbox_valid_foreground4x768_v1'
d3_weight=hf_hub_download(D3_REPOSITORY,'model.safetensors',revision=D3_REVISION)
assert file_sha256(d3_weight)==D3_WEIGHT_SHA256,'DINOv3 checkpoint SHA256 mismatch.'
feature_rows=pd.concat([train_frame,test_frame],ignore_index=True)
def native_view(row):
    im=rgb_source(row['source_path'])
    return im.crop(tuple(row['bbox'])) if row['is_crop'] else im
def d3_prepare(row,mean,std):
    im=native_view(row); width,height=im.size; scale=256/max(width,height)
    nw,nh=max(1,round(width*scale)),max(1,round(height*scale)); dx,dy=(256-nw)//2,(256-nh)//2
    canvas=np.empty((256,256,3),np.uint8);canvas[:]=[124,116,104]
    resized=np.asarray(im.resize((nw,nh),Image.Resampling.BICUBIC));canvas[dy:dy+nh,dx:dx+nw]=resized
    valid=np.zeros((256,256),np.float32);valid[dy:dy+nh,dx:dx+nw]=1.
    border=np.concatenate([resized[0],resized[-1],resized[:,0],resized[:,-1]],axis=0).astype(np.float32)
    center=np.median(border,axis=0);dispersion=np.linalg.norm(border-center,axis=1);foreground=valid.copy()
    if float(np.quantile(dispersion,.75))<30:
        distance=np.linalg.norm(resized.astype(np.float32)-center,axis=2)
        mask=(distance>max(25.,float(np.quantile(dispersion,.95))+10.)).astype(np.uint8)
        mask=cv2.morphologyEx(mask,cv2.MORPH_CLOSE,np.ones((3,3),np.uint8))
        if .03<=float(mask.mean())<=.90:
            foreground[:]=0;foreground[dy:dy+nh,dx:dx+nw]=mask.astype(np.float32);foreground+=.05*valid
    valid_patch=cv2.resize(valid,(16,16),interpolation=cv2.INTER_AREA).reshape(-1)
    foreground_patch=cv2.resize(foreground,(16,16),interpolation=cv2.INTER_AREA).reshape(-1)
    tensor=(canvas.astype(np.float32)/255.-mean[None,None,:])/std[None,None,:]
    return torch.from_numpy(tensor.transpose(2,0,1).copy()),valid_patch,foreground_patch
def d3_descriptor_batch(encoder,prepared):
    images=torch.stack([a[0] for a in prepared]).to(DEVICE)
    tokens=encoder.forward_features(images);patches=tokens[:,encoder.num_prefix_tokens:]
    assert patches.shape[1:]==(256,768)
    valid=torch.as_tensor(np.stack([a[1] for a in prepared]),device=DEVICE);valid/=valid.sum(1,keepdim=True).clamp_min(1e-8)
    foreground=torch.as_tensor(np.stack([a[2] for a in prepared]),device=DEVICE);foreground/=foreground.sum(1,keepdim=True).clamp_min(1e-8)
    global_mean=(patches*valid[:,:,None]).sum(1);object_mean=(patches*foreground[:,:,None]).sum(1)
    object_std=(((patches-object_mean[:,None,:]).square()*foreground[:,:,None]).sum(1).clamp_min(0)).sqrt()
    return torch.cat([F.normalize(a,dim=1) for a in [tokens[:,0],global_mean,object_mean,object_std]],dim=1).cpu().numpy().astype(np.float32)
def hand_descriptor(row):
    im=native_view(row);rgb=np.asarray(im.resize((128,128),Image.Resampling.LANCZOS),dtype=np.uint8)
    hsv=cv2.cvtColor(rgb,cv2.COLOR_RGB2HSV);lab=cv2.cvtColor(rgb,cv2.COLOR_RGB2LAB)
    blocks=[]
    def hist(values,bins,range_):
        h=np.histogram(values,bins=bins,range=range_)[0].astype(np.float32);return h/max(h.sum(),1.)
    for channel,bins,upper in [(0,16,180),(1,8,256),(2,8,256)]:blocks.append(hist(hsv[:,:,channel],bins,(0,upper)))
    joint=np.histogram2d(hsv[:,:,0].ravel(),hsv[:,:,1].ravel(),bins=(8,4),range=((0,180),(0,256)))[0].reshape(-1).astype(np.float32);blocks.append(joint/max(joint.sum(),1.))
    for channel in range(3):blocks.append(hist(lab[:,:,channel],8,(0,256)))
    for yy in [0,64]:
        for xx in [0,64]:
            patch=hsv[yy:yy+64,xx:xx+64]
            for channel,bins,upper in [(0,8,180),(1,4,256),(2,4,256)]:blocks.append(hist(patch[:,:,channel],bins,(0,upper)))
    arr=rgb.astype(np.float32)/255.;blocks.extend([arr.mean((0,1)),arr.std((0,1))])
    h,s,v=hsv[:,:,0],hsv[:,:,1],hsv[:,:,2]
    fractions=[((h>=4)&(h<=30)&(s>=45)&(v>=25)&(v<=225)).mean(),(s<30).mean(),(s>120).mean(),(v<40).mean(),(v>220).mean(),s.std()/255.,v.std()/255.]
    blocks.append(np.array(fractions,np.float32))
    gray=np.asarray(im.convert('L').resize((64,64),Image.Resampling.LANCZOS),dtype=np.uint8)
    blocks.append(hog(gray.astype(np.float32)/255.,orientations=9,pixels_per_cell=(16,16),cells_per_block=(2,2),block_norm='L2-Hys').astype(np.float32))
    blocks.append(hist(local_binary_pattern(gray,8,1,method='uniform'),10,(0,10)))
    return np.concatenate(blocks).astype(np.float32)

cache=WORK_ROOT/'processed/dinov3_hybrid_features.npz'
feature_identity=hashlib.sha256((STATE_SHA256+D3_WEIGHT_SHA256+D3_RECIPE+'native_roi_hand_v1').encode()).hexdigest()
if cache.exists():
    with np.load(cache,allow_pickle=False) as zz:
        assert str(zz['identity'].item())==feature_identity
        d3_features=zz['dino'].copy();hand_features=zz['hand'].copy()
else:
    encoder=timm.create_model('vit_base_patch16_dinov3.lvd1689m',pretrained=True,num_classes=0,pretrained_cfg_overlay={'file':str(d3_weight)}).eval().to(DEVICE)
    if hasattr(encoder,'rope') and hasattr(encoder.rope,'periods'):encoder.rope.periods=encoder.rope.periods.to(torch.bfloat16).to(torch.float32)
    cfg=timm.data.resolve_model_data_config(encoder);mean=np.asarray(cfg['mean'],np.float32);std=np.asarray(cfg['std'],np.float32)
    assert encoder.num_features==768
    records=feature_rows.to_dict('records');chunks=[];batch=8 if DEVICE.type=='cuda' else 2
    # FP32 preserves the established feature recipe; no local-label adaptation of this encoder.
    with torch.inference_mode():
        for start in range(0,len(records),batch):
            chunks.append(d3_descriptor_batch(encoder,[d3_prepare(row,mean,std) for row in records[start:start+batch]]))
            if start%256==0:print('DINOv3 descriptors:',min(start+batch,len(records)),'/',len(records),flush=True)
    d3_features=np.concatenate(chunks);hand_features=np.stack([hand_descriptor(row) for row in records])
    np.savez_compressed(cache,dino=d3_features,hand=hand_features,identity=np.array(feature_identity))
    del encoder;gc.collect();torch.cuda.empty_cache()
assert d3_features.shape==(len(feature_rows),3072) and np.isfinite(d3_features).all() and np.isfinite(hand_features).all()
dino_train=normalize(d3_features[:len(train_frame)]);dino_test=normalize(d3_features[len(train_frame):])
hand_train=hand_features[:len(train_frame)];hand_test=hand_features[len(train_frame):]
DINO_PROVENANCE={'repository':D3_REPOSITORY,'revision':D3_REVISION,'weight_sha256':D3_WEIGHT_SHA256,'recipe':D3_RECIPE,'feature_identity':feature_identity,'train_rows':len(train_frame),'test_rows':len(test_frame),'tree_features':'train-fold PCA of DINO + HSV/Lab/spatial color + HOG/LBP','native_view':'same processed sample membership/ROI/label, native resolution before model resize','timm':timm.__version__,'xgboost':xgboost.__version__}
(WORK_ROOT/'logs/dino_feature_provenance.json').write_text(json.dumps(DINO_PROVENANCE,indent=2),encoding='utf-8')
print('DINO:',dino_train.shape,'handcrafted:',hand_train.shape)

## 9. SVM, group-kNN, Extra Trees, dan XGBoost

RBF-SVM dan kNN berbobot kelompok sumber menjadi baseline. Extra Trees dan XGBoost memakai PCA fitur DINO yang di-fit pada partisi train ditambah fitur warna/tekstur. Pilihan k, bobot tetangga, dan campuran tree dipilih lewat inner grouped validation; outer validation tidak ikut fitting atau pemilihan tersebut. Seluruh model mempertahankan assignment processed. Baseline, tree individual dan campurannya diekspor terpisah. CSV utama awal tersedia sebelum training CNN agar hasil klasik tidak hilang bila sesi berhenti kemudian.

In [ ]:
from collections import Counter
from scipy.special import softmax
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.decomposition import PCA
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_sample_weight
from threadpoolctl import threadpool_limits
K_GRID=(1,3,5,7,11);NEIGHBOR_BLEND_GRID=(0.,.1,.2,.3,.4);TREE_COUNT=600;XGB_ROUNDS=360;INNER_SPLITS=3
TREE_RECIPES=[{'base':1.,'et':0.,'xgb':0.},{'base':.9,'et':.1,'xgb':0.},{'base':.9,'et':0.,'xgb':.1},{'base':.8,'et':.1,'xgb':.1},{'base':.8,'et':.2,'xgb':0.},{'base':.8,'et':0.,'xgb':.2},{'base':.6,'et':.2,'xgb':.2}]
def scores(y,prob):
    pred=prob.argmax(1)
    return {'agreement':float(accuracy_score(y,pred)),'macro_f1':float(f1_score(y,pred,labels=list(range(len(CLASSES))),average='macro',zero_division=0))}
def not_degraded(candidate,baseline):
    return candidate['agreement']>=baseline['agreement'] and candidate['macro_f1']>=baseline['macro_f1'] and (candidate['agreement']>baseline['agreement'] or candidate['macro_f1']>baseline['macro_f1'])
def group_neighbors(qx,fx,fy,fit_groups,k):
    sim=qx@fx.T;k=min(k,len(fx));ix=np.argsort(-sim,axis=1,kind='stable')[:,:k]
    top=np.take_along_axis(sim,ix,axis=1);count=Counter(fit_groups.tolist())
    source_weight=np.array([1./count[g] for g in fit_groups]);w=np.exp((top-top.max(1,keepdims=True))/.05)*source_weight[ix]
    pp=np.column_stack([(w*(fy[ix]==c)).sum(1) for c in range(len(CLASSES))]);return pp/np.maximum(pp.sum(1,keepdims=True),1e-12)
def fit_feature_models(x,h,y,g,qx,qh,seed):
    assert set(y)==set(range(len(CLASSES))),f'Classifier fit split is missing class IDs: {sorted(set(range(len(CLASSES)))-set(y))}'
    svm=OneVsRestClassifier(SVC(C=4.,kernel='rbf',gamma='scale'),n_jobs=1).fit(x,y)
    p_svm=softmax(svm.decision_function(qx),axis=1)
    # PCA is fitted inside the training partition, never on validation/test.
    pca=PCA(n_components=min(192,len(x)-1,x.shape[1]),svd_solver='randomized',random_state=seed)
    xt=np.column_stack([pca.fit_transform(x),h]).astype(np.float32);xq=np.column_stack([pca.transform(qx),qh]).astype(np.float32)
    counts=Counter(g.tolist());source_weights=np.array([1./counts[v] for v in g],np.float32)
    et=ExtraTreesClassifier(n_estimators=TREE_COUNT,min_samples_leaf=2,max_features=.6,class_weight='balanced',n_jobs=min(4,os.cpu_count() or 1),random_state=seed)
    et.fit(xt,y,sample_weight=source_weights)
    backend='cuda' if DEVICE.type=='cuda' else 'cpu'
    xgb=xgboost.XGBClassifier(n_estimators=XGB_ROUNDS,max_depth=4,learning_rate=.04,min_child_weight=2.,subsample=.85,colsample_bytree=.75,reg_lambda=8.,reg_alpha=.1,objective='multi:softprob',num_class=len(CLASSES),tree_method='hist',device=backend,max_bin=128,n_jobs=min(4,os.cpu_count() or 1),random_state=seed,eval_metric='mlogloss')
    weights=compute_sample_weight('balanced',y)*source_weights
    try:xgb.fit(xt,y,sample_weight=weights)
    except xgboost.core.XGBoostError as error:
        if backend=='cuda' and ('cuda' in str(error).lower() or 'gpu' in str(error).lower()):
            print('XGBoost CUDA fallback to CPU:',str(error)[:240],flush=True);xgb.set_params(device='cpu');xgb.fit(xt,y,sample_weight=weights)
        else:raise
    assert np.array_equal(et.classes_,np.arange(len(CLASSES))) and np.array_equal(xgb.classes_,np.arange(len(CLASSES)))
    predictions={'svm':p_svm,'et':et.predict_proba(xq),'xgb':xgb.predict_proba(xq),'neighbors':{k:group_neighbors(qx,x,y,g,k) for k in K_GRID}}
    return predictions,{'svm':svm,'pca':pca,'extra_trees':et,'xgboost':xgb}
def choose_knn_recipe(y,svm,neighbors):
    options=[]
    for k in K_GRID:
        for weight in NEIGHBOR_BLEND_GRID:
            metric=scores(y,(1-weight)*svm+weight*neighbors[k])
            options.append((metric['agreement'],metric['macro_f1'],-weight,-k,k,weight))
    best=max(options);return {'k':int(best[4]),'weight':float(best[5])}
def blend_trees(base,et,xgb,recipe):return recipe['base']*base+recipe['et']*et+recipe['xgb']*xgb
def choose_tree_recipe(y,base,et,xgb):
    baseline=scores(y,base);chosen=TREE_RECIPES[0];best=baseline
    for recipe in TREE_RECIPES[1:]:
        metric=scores(y,blend_trees(base,et,xgb,recipe))
        if not_degraded(metric,baseline) and (metric['agreement'],metric['macro_f1'])>(best['agreement'],best['macro_f1']):chosen=recipe;best=metric
    return dict(chosen)
def valid_inner_folds(y,g,seed):
    # Grouped stratification is approximate. Retry deterministic seeds solely for class coverage.
    for attempt in range(16):
        used_seed=seed+attempt
        cv=StratifiedGroupKFold(n_splits=INNER_SPLITS,shuffle=True,random_state=used_seed)
        candidate=list(cv.split(np.zeros(len(y)),y,g))
        if all(set(y[tr])==set(range(len(CLASSES))) for tr,va in candidate):
            for tr,va in candidate:assert set(g[tr]).isdisjoint(set(g[va]))
            return candidate,used_seed
    raise ValueError('Cannot form source-group inner folds with all classes in each fit split.')
def inner_classical_choice(indices,seed):
    y=pseudo[indices];g=groups[indices];n=len(indices)
    inner,inner_seed=valid_inner_folds(y,g,seed)
    sv=np.full((n,len(CLASSES)),np.nan);et=np.full_like(sv,np.nan);xg=np.full_like(sv,np.nan);neighbors={k:np.full_like(sv,np.nan) for k in K_GRID}
    with threadpool_limits(limits=4):
        for fold,(tr,va) in enumerate(inner):
            assert set(g[tr]).isdisjoint(set(g[va]))
            result,bundle=fit_feature_models(dino_train[indices[tr]],hand_train[indices[tr]],y[tr],g[tr],dino_train[indices[va]],hand_train[indices[va]],seed+fold)
            sv[va]=result['svm'];et[va]=result['et'];xg[va]=result['xgb']
            for k in K_GRID:neighbors[k][va]=result['neighbors'][k]
            del bundle;gc.collect()
    assert np.isfinite(sv).all() and np.isfinite(et).all() and np.isfinite(xg).all()
    neighbor_recipe=choose_knn_recipe(y,sv,neighbors)
    neighbor_recipe['inner_split_seed']=inner_seed
    base=(1-neighbor_recipe['weight'])*sv+neighbor_recipe['weight']*neighbors[neighbor_recipe['k']]
    tree_recipe=choose_tree_recipe(y,base,et,xg)
    return neighbor_recipe,tree_recipe

classic_names=['dino_svm','dino_groupknn','extra_trees','xgboost','dino_tree_hybrid']
classic_oof_probs={name:np.full((len(train_frame),len(CLASSES)),np.nan,np.float32) for name in classic_names};classic_choices=[]
with threadpool_limits(limits=4):
    for fold,(tr,va) in enumerate(folds):
        neighbor_recipe,tree_recipe=inner_classical_choice(tr,SEED+2000+fold)
        result,bundle=fit_feature_models(dino_train[tr],hand_train[tr],pseudo[tr],groups[tr],dino_train[va],hand_train[va],SEED+3000+fold)
        base=(1-neighbor_recipe['weight'])*result['svm']+neighbor_recipe['weight']*result['neighbors'][neighbor_recipe['k']]
        classic_oof_probs['dino_svm'][va]=result['svm'];classic_oof_probs['dino_groupknn'][va]=base
        classic_oof_probs['extra_trees'][va]=result['et'];classic_oof_probs['xgboost'][va]=result['xgb']
        classic_oof_probs['dino_tree_hybrid'][va]=blend_trees(base,result['et'],result['xgb'],tree_recipe)
        classic_choices.append({'fold':fold,'neighbor':neighbor_recipe,'tree':tree_recipe})
        print('Classic nested outer fold',fold,classic_choices[-1],flush=True);del bundle;gc.collect()
classic_summaries=[]
for name,pp in classic_oof_probs.items():
    assert np.isfinite(pp).all() and np.allclose(pp.sum(1),1,atol=1e-4)
    metric=scores(pseudo,pp);classic_summaries.append({'model':name,'proxy_agreement':metric['agreement'],'proxy_macro_f1':metric['macro_f1']})
    pd.DataFrame(pp,columns=CLASSES).assign(sample_id=train_ids,fold=fold_ids,target=train_frame.assigned_class).to_csv(WORK_ROOT/f'validation/oof_{name}.csv',index=False)
baseline_metric=scores(pseudo,classic_oof_probs['dino_groupknn']);hybrid_metric=scores(pseudo,classic_oof_probs['dino_tree_hybrid'])
CLASSIC_SELECTED_NAME='dino_tree_hybrid' if not_degraded(hybrid_metric,baseline_metric) else 'dino_groupknn'
full_neighbor,full_tree=inner_classical_choice(eligible_idx,SEED+4000)
with threadpool_limits(limits=4):
    result,bundle=fit_feature_models(dino_train,hand_train,pseudo,groups,dino_test,hand_test,SEED+5000)
base=(1-full_neighbor['weight'])*result['svm']+full_neighbor['weight']*result['neighbors'][full_neighbor['k']]
classic_final_probabilities={'dino_svm':result['svm'],'dino_groupknn':base,'extra_trees':result['et'],'xgboost':result['xgb'],'dino_tree_hybrid':blend_trees(base,result['et'],result['xgb'],full_tree)}
bundle['xgboost'].save_model(WORK_ROOT/'checkpoints/final_xgboost.json')
joblib.dump({k:v for k,v in bundle.items() if k!='xgboost'},WORK_ROOT/'checkpoints/final_dino_tree_models.joblib',compress=3)
np.savez_compressed(WORK_ROOT/'checkpoints/final_dino_knn_fit.npz',x=dino_train,y=pseudo,groups=groups,classes=np.asarray(CLASSES))
for name,pp in classic_final_probabilities.items():
    write_submission(pp,WORK_ROOT/f'predictions/submission_{name}.csv')
    pd.DataFrame(pp,columns=CLASSES).assign(image_id=test_ids).to_csv(WORK_ROOT/f'predictions/test_probabilities_{name}.csv',index=False)
write_submission(classic_final_probabilities[CLASSIC_SELECTED_NAME],WORK_ROOT/'DSC2026_Submission.csv')
classic_report={'selected_family':CLASSIC_SELECTED_NAME,'outer_folds':N_SPLITS,'inner_folds':INNER_SPLITS,'outer_choices':classic_choices,'full_train_neighbor':full_neighbor,'full_train_tree':full_tree,'tree_count':TREE_COUNT,'xgb_rounds':XGB_ROUNDS,'xgb_actual_device':json.loads(bundle['xgboost'].get_booster().save_config()).get('learner',{}).get('generic_param',{}).get('device'),'pca':'192 components max, fitted independently within every fit split','selection_rule':'train-only inner validation; retain group-kNN baseline unless tree hybrid is nondegraded on paired outer agreement and macro-F1','outer_metrics':classic_summaries,'data_state_sha256':STATE_SHA256,'test_rows_in_fit':0,'probability_note':'SVM softmax margins are ranking scores, not calibrated confidence'}
(WORK_ROOT/'logs/classical_selection.json').write_text(json.dumps(classic_report,indent=2),encoding='utf-8')
display(pd.DataFrame(classic_summaries));print('Classical submission already saved; selected:',CLASSIC_SELECTED_NAME)
del bundle;gc.collect();torch.cuda.empty_cache()

## 10. Evaluasi CNN pada source fold yang sama

Empat CNN dilatih seperti jadwal sebelumnya. Ensemble CNN global dibandingkan dengan baseline klasik menggunakan campuran tetap 80% klasik dan 20% CNN. Campuran hanya dipromosikan bila agreement dan macro-F1 grouped OOF tidak turun dan sedikitnya satu naik. Ini perbandingan internal untuk seleksi model, bukan estimasi untouched final accuracy; tidak ada label atau aturan per ID test.

In [ ]:
oof_probs=dict(classic_oof_probs); proxy_summary=list(classic_summaries)
for name in MODEL_NAMES:
    oof=np.full((len(train_frame),len(CLASSES)),np.nan,dtype=np.float32)
    for fold,(tr,va) in enumerate(folds):
        model,history,checkpoint=fit_one(name,tr,va,FOLD_EPOCHS,f'fold{fold}')
        loader=DataLoader(LabeledDataset(va),batch_size=BATCH_SIZE*2,shuffle=False,num_workers=WORKERS,pin_memory=DEVICE.type=='cuda')
        pp,yy,ids=predict_loader(model,loader); assert ids==[train_ids[j] for j in va]; oof[va]=pp
        del model; gc.collect(); torch.cuda.empty_cache()
    assert np.isfinite(oof).all() and np.allclose(oof.sum(1),1,atol=1e-4)
    oof_probs[name]=oof; pred=oof.argmax(1)
    row={'model':name,'proxy_agreement':float(accuracy_score(pseudo,pred)),'proxy_macro_f1':float(f1_score(pseudo,pred,labels=list(range(len(CLASSES))),average='macro',zero_division=0))}; proxy_summary.append(row)
    pd.DataFrame(oof,columns=CLASSES).assign(sample_id=train_ids,fold=fold_ids,target=train_frame.assigned_class,predicted=[CLASSES[j] for j in pred]).to_csv(WORK_ROOT/f'validation/oof_{name}.csv',index=False)
    pd.DataFrame(confusion_matrix(pseudo,pred,labels=list(range(len(CLASSES)))),index=CLASSES,columns=CLASSES).to_csv(WORK_ROOT/f'validation/confusion_{name}.csv')
    pd.DataFrame(classification_report(pseudo,pred,labels=list(range(len(CLASSES))),target_names=CLASSES,output_dict=True,zero_division=0)).T.to_csv(WORK_ROOT/f'validation/report_{name}.csv')
    print(row)
oof_ensemble=np.mean([oof_probs[name] for name in MODEL_NAMES],axis=0)
proxy_summary.append({'model':'equal_probability_ensemble','proxy_agreement':float(accuracy_score(pseudo,oof_ensemble.argmax(1))),'proxy_macro_f1':float(f1_score(pseudo,oof_ensemble.argmax(1),labels=list(range(len(CLASSES))),average='macro',zero_division=0))})
pd.DataFrame(proxy_summary).to_csv(WORK_ROOT/'validation/proxy_summary.csv',index=False); display(pd.DataFrame(proxy_summary))
fixed_hybrid_oof=.8*classic_oof_probs[CLASSIC_SELECTED_NAME]+.2*oof_ensemble
PROMOTE_NEURAL_BLEND=not_degraded(scores(pseudo,fixed_hybrid_oof),scores(pseudo,classic_oof_probs[CLASSIC_SELECTED_NAME]))
hybrid_metric=scores(pseudo,fixed_hybrid_oof)
proxy_summary.append({'model':'fixed_80classic_20neural','proxy_agreement':hybrid_metric['agreement'],'proxy_macro_f1':hybrid_metric['macro_f1']})
pd.DataFrame(proxy_summary).to_csv(WORK_ROOT/'validation/proxy_summary.csv',index=False)
print('Promote fixed global CNN blend:',PROMOTE_NEURAL_BLEND,'| baseline:',CLASSIC_SELECTED_NAME)

## 11. Refit CNN, inferensi, dan final submission

Refit seluruh 1.886 sampel processed, satu backbone per giliran. Simpan CSV setiap model, ensemble neural, dan campuran klasik-neural. `DSC2026_Submission.csv` mengikuti kebijakan global yang ditetapkan dari train; model tambahan yang tidak lolos guard tidak menggantikan baseline klasik.

In [ ]:
FINAL_EPOCHS=HEAD_EPOCHS+FINETUNE_EPOCHS
model_probabilities={}; final_meta=[]
test_loader=DataLoader(ImageTensor(test_paths),batch_size=BATCH_SIZE*2,shuffle=False,num_workers=WORKERS,pin_memory=DEVICE.type=='cuda')
def write_submission(probabilities,path):
    assert probabilities.shape==(len(test_ids),len(CLASSES)) and np.isfinite(probabilities).all()
    assert np.allclose(probabilities.sum(1),1,atol=1e-4)
    pred=pd.DataFrame({'image_id_norm':test_ids,'waste_category':[CLASSES[j] for j in probabilities.argmax(1)]})
    out=sample[['image_id','image_id_norm']].merge(pred,on='image_id_norm',how='left',validate='one_to_one',sort=False)
    assert out.image_id.tolist()==sample.image_id.tolist()
    assert len(out)==480 and out.waste_category.notna().all() and set(out.waste_category)<=set(CLASSES)
    out=out[['image_id','waste_category']]; out.to_csv(path,index=False)
    return out
for name in MODEL_NAMES:
    model,history,checkpoint=fit_one(name,eligible_idx,None,FINAL_EPOCHS,'final')
    pp,_,_=predict_loader(model,test_loader)
    model_probabilities[name]=pp
    pd.DataFrame(pp,columns=CLASSES).assign(image_id=test_ids).to_csv(WORK_ROOT/f'predictions/test_probabilities_{name}.csv',index=False)
    write_submission(pp,WORK_ROOT/f'predictions/submission_{name}.csv')
    final_meta.append({'model':name,'checkpoint':str(checkpoint.relative_to(WORK_ROOT)),'checkpoint_sha256':file_sha256(checkpoint),'completed_epochs':len(history),'selection':'last epoch of fixed full-train refit; no validation set'})
    del model; gc.collect(); torch.cuda.empty_cache()
cnn_ensemble=np.mean(list(model_probabilities.values()),axis=0)
write_submission(cnn_ensemble,WORK_ROOT/'predictions/submission_neural_ensemble.csv')
hybrid_ensemble=.8*classic_final_probabilities[CLASSIC_SELECTED_NAME]+.2*cnn_ensemble
write_submission(hybrid_ensemble,WORK_ROOT/'predictions/submission_classic_neural_hybrid.csv')
ensemble=hybrid_ensemble if PROMOTE_NEURAL_BLEND else classic_final_probabilities[CLASSIC_SELECTED_NAME]
FINAL_ENSEMBLE_POLICY={'classic_family':CLASSIC_SELECTED_NAME,'cnn_blend_promoted':bool(PROMOTE_NEURAL_BLEND),'weights':{'classic':.8,'cnn_mean':.2} if PROMOTE_NEURAL_BLEND else {'classic':1.,'cnn_mean':0.},'guard':'paired grouped OOF agreement and macro-F1 must both be nondegraded; at least one strict gain; model-selection comparison is not an untouched estimate'}
submission=write_submission(ensemble,WORK_ROOT/'DSC2026_Submission.csv')
pd.DataFrame(ensemble,columns=[f'prob_{c}' for c in CLASSES]).assign(image_id=test_ids).to_csv(WORK_ROOT/'predictions/test_probabilities_ensemble.csv',index=False)
display(submission.head(10)); print('Final CSV:',WORK_ROOT/'DSC2026_Submission.csv')

## 12. Ekspor log, kurva, hash, dan artefak

Manifest merekam label state, descriptor DINOv3, sumber bobot, seluruh evaluasi kandidat, recipe inner/outer, checkpoint, dan kebijakan submission. Semua keluaran disimpan di working dan paket ZIP.

In [ ]:
for name in MODEL_NAMES:
    fig,axs=plt.subplots(1,2,figsize=(12,4))
    for fold in range(N_SPLITS):
        h=pd.read_csv(WORK_ROOT/f'logs/fold{fold}_{name}_history.csv')
        axs[0].plot(h.epoch,h.train_loss,label=f'fold {fold}')
        axs[1].plot(h.epoch,h.val_proxy_macro_f1,label=f'fold {fold}')
    axs[0].set_title(name+' train loss'); axs[1].set_title(name+' validation proxy macro-F1')
    for ax in axs: ax.set_xlabel('epoch'); ax.legend()
    plt.tight_layout(); plt.savefig(WORK_ROOT/f'figures/history_{name}.png',dpi=150); plt.show(); plt.close(fig)
run_manifest={'run_utc':datetime.now(timezone.utc).isoformat(),'environment':ENVIRONMENT,'processing_equivalence':proof,'training_config':TRAIN_CONFIG,'config_sha256':CONFIG_HASH,'proxy_validation':proxy_summary,'final_models':final_meta,'ensemble_policy':FINAL_ENSEMBLE_POLICY,'classical_selection':classic_report,'dino_feature_provenance':DINO_PROVENANCE,'submission_rows':len(submission),'submission_sha256':file_sha256(WORK_ROOT/'DSC2026_Submission.csv'),'submission_label_distribution':submission.waste_category.value_counts().to_dict(),'raw_test_used_for_fit':False,'assignment_fitting_repeated':False,'elapsed_training_seconds':round(time.time()-RUN_START,2)}
pretrained={'resnet50':models.ResNet50_Weights.IMAGENET1K_V2,'efficientnet_v2_s':models.EfficientNet_V2_S_Weights.IMAGENET1K_V1,'convnext_small':models.ConvNeXt_Small_Weights.IMAGENET1K_V1,'swin_v2_t':models.Swin_V2_T_Weights.IMAGENET1K_V1}
run_manifest['pretrained_weight_sources']={}
for name,weight in pretrained.items():
    cached=Path(torch.hub.get_dir())/'checkpoints'/weight.url.rsplit('/',1)[-1]
    run_manifest['pretrained_weight_sources'][name]={'enum':str(weight),'url':weight.url,'sha256':file_sha256(cached) if cached.exists() else None}
(WORK_ROOT/'logs/run_manifest.json').write_text(json.dumps(run_manifest,indent=2),encoding='utf-8')
csv=pd.read_csv(WORK_ROOT/'DSC2026_Submission.csv',dtype={'image_id':str})
assert list(csv.columns)==['image_id','waste_category'] and csv.image_id.tolist()==sample.image_id.tolist() and len(csv)==480
zip_path=Path('/kaggle/working/GelarRasa_GPU_run_artifacts.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as archive:
    for p in WORK_ROOT.rglob('*'):
        if not p.is_file() or p.name.endswith('_resume.pt'): continue
        if 'checkpoints' in p.parts and not p.name.startswith('final_'): continue
        archive.write(p,p.relative_to(WORK_ROOT.parent))
print('CSV:',WORK_ROOT/'DSC2026_Submission.csv')
print('Proof:',WORK_ROOT/'processed/equivalence_report.json')
print('ZIP:',zip_path)